# Review — 1_foundations (100% local, with Ollama)

Review notebook of the **design patterns** covered in `1_lab1` → `5_extra`,
rewritten to run entirely on **free Ollama models** (no API key needed).

## Setup

```bash
ollama pull llama3.2   # small model for plain chat (blocks 1-4)
ollama pull qwen3:8b    # reliable tool-calling model (used in blocks 5-8)
ollama serve            # if not already running
```

> **Why not `llama3.2` for tools?** `llama3.2` (1b/3b) often produces malformed or
> intermittent tool calls. `qwen3:8b` (or `qwen2.5:7b` as an alternative) is far more
> reliable at function calling.

## Index of macro-blocks

| # | Block | Pattern | Source lab |
|---|--------|---------|-----------------|
| 1 | Basic LLM call | `messages` list + OpenAI-compatible client | lab1 |
| 2 | System prompt & memory | Conversation history, the illusion of memory | lab3 |
| 3 | Provider-agnostic client | One SDK (`OpenAI`) for N providers via `base_url` | lab2 |
| 4 | Multi-model + judge | Compare answers + LLM-as-judge (JSON) | lab2 |
| 5 | Tool calling — fundamentals | Tool JSON schema + `if` dispatch | lab3 (step 1-2) |
| 6 | Generalized agent loop | `while` + dispatch via `globals()` (no `if`) | lab3 (step 3) + **lab4** |
| 7 | Digital Twin | Persona from external data + tool with side-effect (lead capture) | lab4 |
| 8 | Visible agent loop | Explicit planning with a "checklist" tool | 5_extra |


In [ ]:
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import json, requests

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

CHAT_MODEL = "llama3.2"     # for plain chat, no tools
TOOL_MODEL = "qwen3:8b"     # for anything using tool calling (blocks 5-8)

def check_model(name):
    try:
        models = {m["id"] for m in requests.get("http://localhost:11434/v1/models", timeout=3).json().get("data", [])}
    except Exception:
        print("Ollama unreachable on localhost:11434 — make sure 'ollama serve' is running.")
        return False
    if not any(m == name or m.startswith(name + ":") for m in models):
        print(f"Model '{name}' not found. Run: ollama pull {name}")
        return False
    return True

check_model(CHAT_MODEL) and check_model(TOOL_MODEL)


## Block 1 — Basic LLM call

**Pattern:** a list of `messages` (OpenAI format) passed to `client.chat.completions.create`.
Every "OpenAI-compatible" provider (Ollama included) accepts the same shape — that's why
you always use the `OpenAI` SDK, even for local models.


In [ ]:
messages = [{"role": "user", "content": "Give me a fun fact in one sentence."}]

response = ollama.chat.completions.create(model=CHAT_MODEL, messages=messages)
print(response.choices[0].message.content)


## Block 2 — System prompt, conversation history, illusion of memory

Three key concepts:

1. **System prompt** — describes the overall context/role of the conversation.
2. **Conversation history** — the whole conversation so far, resent on every call.
3. **Illusion of memory** — every call is *stateless*: the model only "remembers"
   because we resend the entire history each time.

Below: same question, with and without history → different answer.


In [ ]:
# Without history: the model doesn't know the name
messages_no_history = [
    {"role": "system", "content": "You are a snarky assistant."},
    {"role": "user", "content": "What's my name?"}
]
r1 = ollama.chat.completions.create(model=CHAT_MODEL, messages=messages_no_history)
print("Without history:", r1.choices[0].message.content)

# With history: the model "remembers" because we resend everything
messages_with_history = [
    {"role": "system", "content": "You are a snarky assistant."},
    {"role": "user", "content": "My name is Giulio"},
    {"role": "assistant", "content": "Nice to meet you, Giulio."},
    {"role": "user", "content": "What's my name?"}
]
r2 = ollama.chat.completions.create(model=CHAT_MODEL, messages=messages_with_history)
print("With history:", r2.choices[0].message.content)


## Block 3 — Provider-agnostic client

**Pattern:** every LLM provider (OpenAI, Anthropic, Gemini, Groq, Ollama, ...) exposes an
OpenAI-compatible endpoint. Only `base_url` (and `api_key`) change — the rest of the code
stays identical. This is what makes swapping a paid model for a free local one trivial.


In [ ]:
# The exact same code works with any provider — only the client/base_url changes.
# We only keep Ollama here (free, local); the others require a paid API key.

CLIENTS = {
    "ollama": OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"),
    # "openai": OpenAI(),  # requires OPENAI_API_KEY
    # "groq":   OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY")),
}

def ask(provider, model, question):
    client = CLIENTS[provider]
    resp = client.chat.completions.create(model=model, messages=[{"role": "user", "content": question}])
    return resp.choices[0].message.content

print(ask("ollama", CHAT_MODEL, "In one word: what is the meaning of life?"))


## Block 4 — Multi-model comparison + LLM-as-judge

**Pattern:** the same question to N different models, then a "judge" model evaluates and
ranks the answers, replying in **pure JSON** (parsed with `json.loads`).

Useful whenever you need to improve the reliability of an LLM answer by aggregating multiple sources.


In [ ]:
question = "In one sentence: what is the fundamental difference between intelligence and knowledge?"

competitors, answers = [], []

def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(f"**{model_name}**: {answer}"))

for model_name in [CHAT_MODEL, TOOL_MODEL]:
    if check_model(model_name):
        resp = ollama.chat.completions.create(model=model_name, messages=[{"role": "user", "content": question}])
        record(model_name, resp.choices[0].message.content)


In [ ]:
together = ""
for i, answer in enumerate(answers):
    together += f"# Response from competitor {i+1}\n\n{answer}\n\n"

judge_prompt = f'''You are judging a competition between {len(competitors)} competitors on this question:

{question}

Evaluate clarity and strength of argument, and rank them from best to worst.
Respond ONLY with JSON in this format, no markdown:
{{"results": ["best competitor number", "second best", ...]}}

Responses:

{together}'''

judge_resp = ollama.chat.completions.create(model=TOOL_MODEL, messages=[{"role": "user", "content": judge_prompt}])
ranks = json.loads(judge_resp.choices[0].message.content)["results"]
for i, r in enumerate(ranks):
    print(f"Rank {i+1}: {competitors[int(r)-1]}")


## Block 5 — Tool calling: the fundamentals

**Pattern:** a tool is described with a JSON schema (name, description, parameters). If the
model decides to use it, it replies with `finish_reason == "tool_calls"` instead of text.

The simplest dispatch is an explicit `if` per tool — it works, but doesn't scale
(see Block 6).


In [ ]:
def record_email_tool(email):
    print(f"[TOOL] email recorded: {email}")
    return "Email recorded"

record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record the email address provided by a user",
    "parameters": {
        "type": "object",
        "properties": {"email": {"type": "string", "description": "The user's email address"}},
        "required": ["email"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": record_email_tool_json}]

def chat_with_if_dispatch(message, history):
    messages = [{"role": "system", "content": "You are an assistant. If the user provides an email, record it with the tool."}] \
        + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_call = msg.tool_calls[0]
        email = json.loads(tool_call.function.arguments).get("email")
        record_email_tool(email)  # <-- the hardcoded IF: only works because there's a single tool
        messages.append(msg)
        messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = ollama.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

print(chat_with_if_dispatch("My email is mario@test.com", []))


## Block 6 — Generalized agent loop (the lab4 pattern)

Two changes from Block 5, exactly as in lab3→lab4:

1. `if` → **`while`**: the model can call tools multiple times in a row before replying (a real agent loop).
2. Dispatch via **`globals()`** instead of an `if/elif` per tool: iterate over `tool_calls`,
   look up the Python function by name in `globals()`, and call it with `**arguments`. Adding a
   new tool doesn't require touching the dispatcher — just write the function and its JSON schema.

This is the pattern behind the Digital Twin (Block 7) and any agent with multiple tools.


In [ ]:
def record_unknown_question(question):
    print(f"[TOOL] unanswered question recorded: {question}")
    return "OK"

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this to record a question you couldn't answer",
    "parameters": {
        "type": "object",
        "properties": {"question": {"type": "string", "description": "The unanswered question"}},
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": record_email_tool_json},
    {"type": "function", "function": record_unknown_question_json},
]

def handle_tool_calls(tool_calls):
    # Generic dispatcher: no if/elif, works with any number of tools.
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "Tool not found"
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

def agent_loop(message, history, system_prompt):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        messages.extend(handle_tool_calls(msg.tool_calls))
        response = ollama.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

demo_system = "You are an assistant. If the user leaves an email, record it; if you can't answer something, record the question."
print(agent_loop("I don't know who to ask, my email is mario@test.com", [], demo_system))


## Block 7 — Digital Twin (the lab4 pattern)

**Design pattern:** an assistant that *plays a persona* by reading external data
(LinkedIn PDF + `summary.txt`) injected into the system prompt, with behavior rules
(stay on topic, don't make things up) and tools with **real-world side-effects**:

- `record_user_details` → captures a lead (email/name/notes) from someone wanting to be contacted
- `record_unknown_question` → tracks knowledge gaps so the twin can be improved later

In the original lab the side-effect is a Pushover notification to a phone; here we
replace it with a `print` to stay 100% local — the structure of the pattern is identical.


In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = "".join(page.extract_text() or "" for page in reader.pages)

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

twin_system_prompt = f'''
# Role
You are the digital twin of a person, chatting with visitors of their website.
Only answer questions about career, background, skills and experience.

# Person's details
{summary}

# LinkedIn context
{linkedin}

# Rules
Be professional. If the question is off-topic, steer the conversation back to professional matters.
If the user wants to be contacted, ask for their email and use the tool to record it.
If you don't know the answer, use the tool to record the question and say so honestly. Never make things up.
'''

def record_user_details(email, name="not provided", notes="not provided"):
    print(f"[LEAD] {name} <{email}> — notes: {notes}")
    return "OK"

record_user_details_json = {
    "name": "record_user_details",
    "description": "Record that a user wants to be contacted, with their email",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The user's email"},
            "name": {"type": "string", "description": "The user's name, if provided"},
            "notes": {"type": "string", "description": "Useful context about the conversation"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
]

print(agent_loop("Hi, tell me a bit about yourself", [], twin_system_prompt))


In [ ]:
# For the interactive UI (optional, opens in the browser):
# import gradio as gr
# gr.ChatInterface(lambda message, history: agent_loop(message, history, twin_system_prompt)).launch(inbrowser=True)


## Block 8 — Visible agent loop: planning with a checklist (the 5_extra pattern)

**Design pattern:** make the agent's step-by-step "reasoning" explicit by giving it a
**planning** tool (`create_checklist`) and a **progress** tool (`mark_complete`), instead of
leaving everything implicit inside a single reply. The same `agent_loop` from Block 6
handles all of it — only the tools change.


In [ ]:
checklist, completed = [], []

def get_checklist_report():
    lines = [f"[{'x' if completed[i] else ' '}] {item}" for i, item in enumerate(checklist)]
    report = "\n".join(lines)
    print(report)
    return report

def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    print(completion_notes)
    return get_checklist_report()

create_checklist_json = {
    "name": "create_checklist",
    "description": "Create the checklist of steps to solve the problem",
    "parameters": {
        "type": "object",
        "properties": {"descriptions": {"type": "array", "items": {"type": "string"}}},
        "required": ["descriptions"],
        "additionalProperties": False
    }
}
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark the step at the given position (1-based) as complete",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {"type": "integer", "description": "1-based index of the step"},
            "completion_notes": {"type": "string", "description": "How it was completed"}
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": create_checklist_json},
    {"type": "function", "function": mark_complete_json},
]

planning_system = '''
You are given a problem. First use the create_checklist tool to plan the steps,
then mark_complete for each step as you solve it. Only reply after using the
tools, with the final solution.
'''

checklist, completed = [], []
print(agent_loop(
    "A train leaves Boston at 2:00 pm traveling 60 mph. Another train leaves New York at "
    "3:00 pm traveling 80 mph toward Boston. When do they meet?",
    [], planning_system
))


## Pattern recap

| Pattern | Key idea |
|---|---|
| Messages list | Universal OpenAI-compatible format for LLM chat |
| System prompt + history | The model is stateless; "memory" lives entirely in the payload we resend |
| Provider-agnostic client | Same SDK, only `base_url`/`api_key` change |
| LLM-as-judge | Multiple answers + one model comparing them in JSON |
| Tool JSON schema | The model chooses to call a function, it doesn't execute it itself |
| `if` → `while` | From "one tool, once" to a real agent loop |
| Dispatch via `globals()` | Add tools without touching the dispatcher |
| Persona from external data | System prompt built from PDF/text for a digital twin |
| Tool with side-effect | A tool can act in the real world (notification, DB, lead) |
| Explicit planning | Checklist tool to make step-by-step reasoning visible |

To review the original paid version (OpenAI), compare with `3_lab3.ipynb` and `4_lab4.ipynb`.
